In [ ]:
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import time

In [ ]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport narrowed param ranges"
mat_path = transport_dir / "all_trial_bkt_data.mat"

In [ ]:
# Reused from the sibling percentile-BKT notebooks -- keep in sync if those
# notebooks' loader changes.
def load_all_trial_bkt_data(mat_path, ps_indices=None):
    param_sets = {}
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data']  # (1, N) object refs
        indices = ps_indices if ps_indices is not None else range(atbd.shape[1])
        for i in indices:
            ps_grp = f[atbd[0, i]]
            p0 = float(ps_grp['p0'][()].flat[0])
            td = ps_grp['trial_data']
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                trial_data.append(arr.flatten() if arr.dtype == np.float64 else np.array([], dtype=np.float64))
            param_sets[i] = {'p0': p0, 'trial_data': trial_data}
    return param_sets, weights


def recover_theta(w):
    disc = 1*w[0] + 2.1*(1 - w[0])           # [1, 2.1]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])  # [0.01e-3, 0.3e-3]
    lr   = 0.3*w[2] + 0.5*(1 - w[2])         # [0.3, 0.5]
    return disc, lr, b


def quantile_bkt(bkt, p, Np):
    """pth-percentile breakthrough time for one trial from its raw particle
    arrival times (order statistic), or None if undefined. Copied from
    [per p] distrib of pth percentile bkt.ipynb."""
    k = int(np.floor(p * Np))
    if k < len(bkt):
        return bkt[k]
    return None


def logit(p, eps=1e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

In [ ]:
# Same connectivity filter as the sibling notebooks.
P0_THRESHOLD = 0.9
Np = 1000

with h5py.File(mat_path, 'r') as f:
    n_param_sets = f['all_trial_bkt_data'].shape[1]

all_param_sets, weights = load_all_trial_bkt_data(mat_path, range(n_param_sets))
USABLE_PS = sorted(i for i, ps in all_param_sets.items() if ps['p0'] < P0_THRESHOLD)
ps_by_p0 = sorted(USABLE_PS, key=lambda i: all_param_sets[i]['p0'])
print(f"{len(USABLE_PS)} / {n_param_sets} param sets usable (p0 < {P0_THRESHOLD})")

In [ ]:
# --- New approach: learn the joint CDF surface F(theta, p, log t) directly,
# instead of fitting a parametric family (2-Gaussian mixture / skew-logistic)
# at each (theta,p) and training a network to predict that family's
# coefficients. Motivated by the joint-surface plots in the sibling notebook
# ("[per p] joint CDF surface F(theta,p,logt).ipynb"), which show the whole
# family of percentile-BKT CDFs forms one smooth, coherent surface in
# (p, log t) rather than a set of unrelated curves -- suggesting a single
# network that takes (theta, p, log t) as input and outputs F directly
# should be able to learn that surface without needing an intermediate
# distributional-family assumption at all.
#
# Training data: for each (theta,p) group (same p_vals grid used
# elsewhere), the empirical CDF is exact at the trials' own order
# statistics -- (log10(t_i), ecdf_i) for i=1..n, ecdf_i=(i-0.5)/n -- no
# parametric fitting step in between. Flatten ALL of these (theta,p,log t)
# -> ecdf rows, across every usable group, into one big supervised
# regression dataset. Each group's n order statistics are highly redundant
# (adjacent sorted values differ by very little in ecdf), so each group is
# subsampled down to at most MAX_PER_GROUP evenly-spaced-by-RANK points
# (preserves the full ecdf range 0->1 and the tail behavior, just drops
# redundant middle density) to keep the flattened dataset a manageable size
# for training.
p_vals = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3,
                    0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
MIN_SAMPLES = 20
MAX_PER_GROUP = 80


def group_ecdf(ps_idx, p):
    """Empirical (log10(t), ecdf) order-statistic pairs for one (theta,p)
    group -- the raw ingredient both this cell's flattened training set and
    the cross-model sse comparison cell are built from, factored out here so
    that shared logic (pull per-trial percentile-BKT values, drop Nones,
    apply MIN_SAMPLES, sort, compute plotting-position ecdf) lives in one
    place instead of two. Returns None if too few samples; otherwise
    (log_vals, ecdf), both length n, FULL (not yet subsampled) -- callers
    that want MAX_PER_GROUP-subsampled rows (like the training-data loop
    below) subsample after calling this; callers that want the full
    empirical curve (like the sse comparison) use it as-is."""
    ps = all_param_sets[ps_idx]
    vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
    vals = np.array([v for v in vals if v is not None])
    if len(vals) < MIN_SAMPLES:
        return None
    log_vals = np.sort(np.log10(vals))
    n = len(log_vals)
    ecdf = (np.arange(1, n + 1) - 0.5) / n
    return log_vals, ecdf


rows = []
for ps_idx in USABLE_PS:
    disc, lr, b = recover_theta(weights[ps_idx])
    for p in p_vals:
        result = group_ecdf(ps_idx, p)
        if result is None:
            continue
        log_vals, ecdf = result
        n = len(log_vals)
        if n > MAX_PER_GROUP:
            keep = np.linspace(0, n - 1, MAX_PER_GROUP).round().astype(int)
            keep = np.unique(keep)
            log_vals, ecdf = log_vals[keep], ecdf[keep]
        for lt, f in zip(log_vals, ecdf):
            rows.append((ps_idx, disc, lr, b, p, lt, f))

flat_df = pd.DataFrame(rows, columns=['paramset_idx', 'discspacing', 'length_ratio', 'b', 'p', 'logt', 'ecdf'])
print(f"Flattened dataset: {len(flat_df)} rows from {flat_df['paramset_idx'].nunique()} param sets "
      f"x up to {len(p_vals)} p_vals x up to {MAX_PER_GROUP} order statistics/group")

In [ ]:
# Inputs: (discspacing, length_ratio, b, logit(p), log10(t)) -- FIVE
# continuous inputs this time (vs. the 4 used in the sibling notebooks,
# which only needed theta+p since t was handled analytically by whichever
# parametric CDF family). Output: a single scalar F in [0,1] via a sigmoid
# last layer (guarantees a valid-range CDF value for any input, same spirit
# as the constrained-output-transform trick used elsewhere, just architecturally
# built in this time instead of applied post-hoc).
#
# Train/val split is BY PARAM SET, not by row -- a lesson learned the hard
# way in the skew-logistic notebook's monotonicity experiment: a row-random
# split lets nearly every theta appear in both train and val (just at
# different p/log t), which is an easier "interpolate for an already-seen
# theta" task, not genuine generalization to new theta. Splitting by whole
# param sets is the honest test, and comparable to how the monotonicity
# experiment's model_nopenalty/model_mono were evaluated.
FEATURE_COLS = ['discspacing', 'length_ratio', 'b', 'p', 'logt']

X_all = np.column_stack([
    flat_df['discspacing'].values, flat_df['length_ratio'].values, flat_df['b'].values,
    logit(flat_df['p'].values), flat_df['logt'].values,
]).astype(np.float64)
y_all = flat_df['ecdf'].values.astype(np.float64)

X_mean, X_std = X_all.mean(axis=0), X_all.std(axis=0)


def normalize_X(X):
    return (X - X_mean) / X_std


X_norm = normalize_X(X_all).astype(np.float32)
y_all = y_all.astype(np.float32)

rng = np.random.default_rng(42)
all_ps = flat_df['paramset_idx'].unique()
val_ps = set(rng.choice(all_ps, size=max(1, int(0.2 * len(all_ps))), replace=False))
is_val = flat_df['paramset_idx'].isin(val_ps).values

X_train, y_train = X_norm[~is_val], y_all[~is_val]
X_val, y_val = X_norm[is_val], y_all[is_val]
print(f"train: {len(X_train)} rows ({(~is_val).sum() and flat_df.loc[~is_val, 'paramset_idx'].nunique()} param sets)  "
      f"val: {len(X_val)} rows ({flat_df.loc[is_val, 'paramset_idx'].nunique()} param sets)")


def build_model():
    l2_reg = 9e-5
    return keras.Sequential([
        keras.layers.Input(shape=(5,)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(1, activation="sigmoid"),
    ])


def lr_by_epoch(epoch):
    if epoch < 80:
        return 1e-3
    elif epoch < 120:
        return 5e-4
    else:
        return 1e-4

In [ ]:
# --- Unconstrained baseline: plain MSE regression, no physics penalty ---
model_plain = build_model()
model_plain.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
t0 = time.time()
history_plain = model_plain.fit(
    X_train, y_train, validation_data=(X_val, y_val), epochs=150, batch_size=512,
    callbacks=[keras.callbacks.LearningRateScheduler(lr_by_epoch, verbose=0)], verbose=0)
print(f"[{time.time()-t0:.0f}s] plain model -- final train loss: {history_plain.history['loss'][-1]:.5f}  "
      f"val loss: {history_plain.history['val_loss'][-1]:.5f}")
model_plain.save(transport_dir / 'direct_cdf_surface_model_plain.keras')

In [ ]:
# --- Physics-constrained version: same data/split/architecture, plus two
# soft penalties enforced via autodiff on the network's own inputs
# (double-backprop / "collocation" style, as in physics-informed NNs):
#
#   dF/d(log t) >= 0   -- F must be non-decreasing in t (valid CDF)
#   dF/dp       <= 0   -- higher-percentile BKT distributions lie to the
#                         right of lower-percentile ones (physical ordering
#                         of order statistics), matching the corrected sign
#                         convention established in the skew-logistic
#                         notebook's monotonicity experiment: for p_i<p_j,
#                         F_{p_i}(t) >= F_{p_j}(t) everywhere.
#
# Both derivatives are taken directly on the Z-SCORED input columns (logit(p)
# and log t, both already z-scored into columns 3 and 4 of X_train) rather
# than converting back to physical units first -- unnecessary extra work,
# since z-scoring (x-mean)/std is a strictly increasing function of x for
# std>0, so the SIGN of dF/d(normalized column) is identical to the sign of
# dF/d(physical quantity). Only the sign matters for a hinge penalty.
#
# Unlike the skew-logistic notebook's monotonicity experiment -- which had
# to batch by param set and compare ADJACENT p rows within a batch, because
# that model only ever produced a CDF via a fitted parametric family evaluated
# at a handful of discrete p's -- this network takes (p, log t) as direct
# continuous inputs, so the constraint can be checked POINTWISE at each
# training row via its own input gradient. No special batching needed: plain
# row-shuffled minibatches work fine, exactly like the unconstrained baseline.
LAM_T = 1.0
LAM_P = 1.0
IDX_P, IDX_LOGT = 3, 4  # columns of X_train corresponding to logit(p), logt (both z-scored)


@tf.function
def train_step(model, opt, X_batch, y_batch):
    with tf.GradientTape() as outer_tape:
        with tf.GradientTape() as inner_tape:
            inner_tape.watch(X_batch)
            F_pred = model(X_batch, training=True)
        dF_dX = inner_tape.gradient(F_pred, X_batch)  # (batch, 5), normalized-input-space partials

        data_loss = tf.reduce_mean(tf.square(y_batch - tf.squeeze(F_pred, axis=1)))
        penalty_t = tf.reduce_mean(tf.nn.relu(-dF_dX[:, IDX_LOGT]))  # want dF/d(logt) >= 0
        penalty_p = tf.reduce_mean(tf.nn.relu(dF_dX[:, IDX_P]))      # want dF/d(logit p) <= 0
        loss = data_loss + LAM_T * penalty_t + LAM_P * penalty_p

    grads = outer_tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return data_loss, penalty_t, penalty_p


def train_constrained(X_train, y_train, X_val, y_val, epochs=150, batch_size=512, verbose=0):
    model = build_model()
    opt = keras.optimizers.Adam(1e-3)
    n = len(X_train)
    history = {'loss': [], 'val_loss': [], 'penalty_t': [], 'penalty_p': []}
    rng_local = np.random.default_rng(1)

    for epoch in range(epochs):
        opt.learning_rate.assign(lr_by_epoch(epoch))
        order = rng_local.permutation(n)
        ep_loss, ep_pt, ep_pp = [], [], []
        for i in range(0, n, batch_size):
            idx = order[i:i + batch_size]
            Xb = tf.constant(X_train[idx])
            yb = tf.constant(y_train[idx])
            dl, pt, pp = train_step(model, opt, Xb, yb)
            ep_loss.append(float(dl)); ep_pt.append(float(pt)); ep_pp.append(float(pp))

        val_pred = model(X_val, training=False).numpy().squeeze(axis=1)
        val_loss = float(np.mean((y_val - val_pred) ** 2))
        history['loss'].append(np.mean(ep_loss))
        history['val_loss'].append(val_loss)
        history['penalty_t'].append(np.mean(ep_pt))
        history['penalty_p'].append(np.mean(ep_pp))
        if verbose and epoch % 25 == 0:
            print(f"epoch {epoch}: loss={history['loss'][-1]:.5f} val_loss={val_loss:.5f} "
                  f"pen_t={history['penalty_t'][-1]:.5f} pen_p={history['penalty_p'][-1]:.5f}")

    return model, history


t0 = time.time()
model_constrained, history_constrained = train_constrained(X_train, y_train, X_val, y_val, epochs=150, verbose=0)
print(f"[{time.time()-t0:.0f}s] constrained model -- final train loss: {history_constrained['loss'][-1]:.5f}  "
      f"val loss: {history_constrained['val_loss'][-1]:.5f}  "
      f"penalty_t: {history_constrained['penalty_t'][-1]:.5f}  penalty_p: {history_constrained['penalty_p'][-1]:.5f}")
model_constrained.save(transport_dir / 'direct_cdf_surface_model_constrained.keras')

In [ ]:
# --- Cross-model comparison metric: same "CDF-space sse against the FULL
# (non-subsampled) empirical order statistics per group" reported in all
# three sibling notebooks -- lets this approach's fit quality be compared
# apples-to-apples against the PDF/MLE baseline (mean sse 0.1015), the
# CDF/2-erf mixture (0.0386), and the CDF/skew-logistic (0.0928) models.
def group_sse(predict_fn):
    sses = []
    for ps_idx in USABLE_PS:
        disc, lr, b = recover_theta(weights[ps_idx])
        for p in p_vals:
            result = group_ecdf(ps_idx, p)  # full (non-subsampled) order statistics -- see cell-4
            if result is None:
                continue
            log_vals, ecdf = result
            n = len(log_vals)
            X_g = normalize_X(np.column_stack([
                np.full(n, disc), np.full(n, lr), np.full(n, b), np.full(n, logit(p)), log_vals,
            ])).astype(np.float32)
            F_pred = predict_fn(X_g)
            sses.append(np.sum((F_pred - ecdf) ** 2))  # same convention as sibling notebooks: sum of squared residuals over that group's own (full, non-subsampled) points
    return np.array(sses)


sse_plain = group_sse(lambda X: model_plain.predict(X, verbose=0).squeeze(axis=1))
sse_constrained = group_sse(lambda X: model_constrained(X, training=False).numpy().squeeze(axis=1))

print("Direct-CDF-surface NN, PLAIN (unconstrained):       sse mean=%.4f  median=%.4f  max=%.4f" %
      (sse_plain.mean(), np.median(sse_plain), sse_plain.max()))
print("Direct-CDF-surface NN, CONSTRAINED (physics penalty): sse mean=%.4f  median=%.4f  max=%.4f" %
      (sse_constrained.mean(), np.median(sse_constrained), sse_constrained.max()))
print("\nFor reference (from the sibling notebooks' own conclusion cells):")
print("  PDF/MLE baseline (2-Gaussian mixture, MLE):      sse mean=0.1015")
print("  CDF/2-erf mixture (2-Gaussian mixture, CDF-LS):  sse mean=0.0386")
print("  CDF/skew-logistic (single S-curve, CDF-LS):      sse mean=0.0928")

In [ ]:
# --- Monotonicity violation check on a fine grid, for a few representative
# param sets -- same style of check as the skew-logistic notebook's
# before/after comparison, but here checking the ACTUAL constraint the
# training penalty targets (sign of finite-difference slopes in p and in
# log t across a fine grid), for both the plain and constrained models.
n_ps = len(ps_by_p0)
REP_PS_IDX = [ps_by_p0[0], ps_by_p0[n_ps // 4], ps_by_p0[n_ps // 2],
              ps_by_p0[3 * n_ps // 4], ps_by_p0[-1]]
P_GRID = np.linspace(0.02, 0.95, 60)


def compute_grid(predict_fn, ps_idx):
    ps = all_param_sets[ps_idx]
    disc, lr, b = recover_theta(weights[ps_idx])
    all_logvals = []
    for p in [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        if len(vals):
            all_logvals.append(np.log10(vals))
    all_logvals = np.concatenate(all_logvals)
    t_lo, t_hi = all_logvals.min() - 0.3, all_logvals.max() + 0.3
    T_GRID = np.linspace(t_lo, t_hi, 150)

    Tm, Pm = np.meshgrid(T_GRID, P_GRID)  # both (n_p, n_t)
    X_grid = normalize_X(np.column_stack([
        np.full(Tm.size, disc), np.full(Tm.size, lr), np.full(Tm.size, b),
        logit(Pm.ravel()), Tm.ravel(),
    ])).astype(np.float32)
    Z = predict_fn(X_grid).reshape(Tm.shape)
    return T_GRID, Z


def violation_rates(predict_fn):
    frac_t_viol, frac_p_viol = [], []
    for ps_idx in REP_PS_IDX:
        T_GRID, Z = compute_grid(predict_fn, ps_idx)
        dZ_dt = np.diff(Z, axis=1)   # along log t (axis=1) -- want >= 0
        dZ_dp = np.diff(Z, axis=0)   # along p (axis=0) -- want <= 0
        frac_t_viol.append((dZ_dt < -1e-6).mean())
        frac_p_viol.append((dZ_dp > 1e-6).mean())
    return np.mean(frac_t_viol), np.mean(frac_p_viol)


t_viol_plain, p_viol_plain = violation_rates(lambda X: model_plain.predict(X, verbose=0).squeeze(axis=1))
t_viol_cons, p_viol_cons = violation_rates(lambda X: model_constrained(X, training=False).numpy().squeeze(axis=1))

print("Fraction of adjacent grid-cell steps violating each constraint (avg over 5 representative param sets):")
print(f"  PLAIN:       dF/d(logt)<0 (should be >=0): {100*t_viol_plain:.2f}%   "
      f"dF/dp>0 (should be <=0): {100*p_viol_plain:.2f}%")
print(f"  CONSTRAINED: dF/d(logt)<0 (should be >=0): {100*t_viol_cons:.2f}%   "
      f"dF/dp>0 (should be <=0): {100*p_viol_cons:.2f}%")

In [ ]:
# --- Visual comparison: joint CDF surface, plain vs constrained, for a
# couple of representative param sets ---
CONTOUR_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
COMPARE_PS_IDX = [REP_PS_IDX[0], REP_PS_IDX[-1]]  # best- and worst-connected

pdf_path = transport_dir / '[per p] direct CDF surface NN, plain vs constrained.pdf'
with PdfPages(pdf_path) as pdf:
    for ps_idx in COMPARE_PS_IDX:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])

        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        for ax, predict_fn, name in zip(
            axes,
            [lambda X: model_plain.predict(X, verbose=0).squeeze(axis=1),
             lambda X: model_constrained(X, training=False).numpy().squeeze(axis=1)],
            ['plain (unconstrained)', 'constrained (physics penalty)'],
        ):
            T_GRID, Z = compute_grid(predict_fn, ps_idx)
            mesh = ax.pcolormesh(T_GRID, P_GRID, Z, shading='auto', cmap='viridis', vmin=0, vmax=1)
            cs = ax.contour(T_GRID, P_GRID, Z, levels=CONTOUR_LEVELS, colors='white', linewidths=1.0, alpha=0.8)
            ax.clabel(cs, fmt='%.1f', fontsize=7, colors='white')
            tick_lo, tick_hi = int(np.floor(T_GRID.min())), int(np.ceil(T_GRID.max()))
            ticks = np.arange(tick_lo, tick_hi + 1)
            ax.set_xticks(ticks)
            ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
            ax.set_xlim(T_GRID.min(), T_GRID.max())
            ax.set_xlabel('breakthrough time (s)')
            ax.set_ylabel('percentile $p$')
            ax.set_title(name, fontsize=12)
        fig.colorbar(mesh, ax=axes, pad=0.02, shrink=0.85, label='CDF $F$')
        fig.suptitle(f'Param set {ps_idx}  disc={disc:.2f}, r={lr:.2f}, b={b:.2e}, p0={ps["p0"]:.3f}', fontsize=14)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}  ({len(COMPARE_PS_IDX)} pages)')

In [ ]:
# --- Conclusion ---
#
# Fit quality (cell-8), sse across all 1479 (theta,p) groups, vs. the three
# sibling parametric-family models:
#   PDF/MLE baseline (2-Gaussian mixture, MLE):          mean 0.1015
#   CDF/2-erf mixture (2-Gaussian mixture, CDF-LS):       mean 0.0386
#   CDF/skew-logistic (single S-curve, CDF-LS):           mean 0.0928
#   Direct surface NN, plain:        mean 3.5354  median 0.1128  max 193.3
#   Direct surface NN, constrained:  mean 4.3565  median 0.0391  max 217.7
# The MEDIAN is genuinely competitive -- constrained's median (0.0391) ties
# the best parametric model (CDF/2-erf mixture, 0.0386), and plain's median
# (0.1128) is in the same range as the PDF/MLE baseline. But the MEAN is
# 30-40x worse than every parametric model, entirely because of a small
# number of catastrophic outliers (max sse ~193-218, vs ~0.2-0.8 for the
# parametric models). Diagnosed directly: EVERY one of the 10 worst rows
# (both models) is the SAME single held-out param set (idx 88) -- its
# predicted F barely moves (stays within [0.003, 0.87]) across the entire
# log-t domain, for every p. This isn't an out-of-range extrapolation
# problem: param set 88's theta (disc=1.29, r=0.40, b=2.3e-5) sits well
# inside the training thetas' range on every coordinate individually. It's
# a genuine single-theta interpolation failure for this particular
# train/val split and network initialization -- plausibly because this
# model has to learn the full 2D (p, log t) -> F SHAPE for held-out theta
# purely by interpolating in a 3D theta neighborhood, a harder generalization
# demand than the parametric-family models face (they only need to
# interpolate 5, or 3, smooth summary numbers per theta). This is the real,
# structural risk of this approach vs. the sibling models -- worth being
# upfront about rather than only reporting the (flattering) median.
#
# Training itself (cell-6/7): both models trained in <30s (150 epochs) on
# the 118,320-row flattened dataset -- an order of magnitude faster than the
# custom monotonicity-penalized loop in the skew-logistic notebook, because
# the constraint here is POINTWISE per training row (no batching-by-param-set
# needed), unlike that notebook's paired-rows-within-a-batch approach.
#
# Monotonicity (cell-9, fine-grid check on 5 representative param sets):
#   PLAIN:       dF/d(logt)<0 violations 0.00%   dF/dp>0 violations 0.07%
#   CONSTRAINED: dF/d(logt)<0 violations 0.00%   dF/dp>0 violations 0.18%
# Both models are already ~monotonic in log t even with NO penalty (0.00%
# both ways) -- unsurprising, since regressing directly onto empirical CDF
# targets (which ARE monotonic in t by construction) gives the network a
# strong implicit bias toward a monotonic-in-t fit, unlike the parametric-
# family models where t-monotonicity was guaranteed analytically by the CDF
# formula itself, not learned. The p-direction is more interesting: the
# CONSTRAINED model shows MORE violations on this grid check (0.18% vs
# 0.07%) despite its own training penalty (cell-7) converging to ~0.00000 by
# the end of training -- i.e. the penalty was satisfied almost everywhere it
# was actually CHECKED (real training-row locations), but this grid check
# also probes many (theta,p,logt) combinations that were never a real
# training row and never had the penalty applied there. This is exactly the
# "no collocation points" limitation noted below, not a contradiction --
# and visually (cell-10's PDF), both models' surfaces for the two inspected
# param sets look clean and smoothly monotonic either way, at least as good
# as the PDF/MLE-baseline notebook's own joint-surface plots for the same
# param sets.
#
# Net takeaway: promising but not yet a clean win. Typical-case fit quality
# (median) already matches the best parametric model, achieved with a
# genuinely simpler idea (no distributional-family assumption at all) and
# faster training -- but the mean is dragged down hard by at least one
# outright generalization failure on a held-out theta, and the physics
# penalty as implemented here (pointwise-only, no collocation points) didn't
# demonstrably improve monotonicity on a held-out fine grid despite driving
# its own training objective to ~0. Both issues point to the same fix:
# broaden where the model/penalty are actually evaluated during training --
# collocation points for the penalty, and likely more training data
# (raise MAX_PER_GROUP, or train on multiple random splits and check
# whether param-set-88-style failures are split-specific or systematic)
# for the outlier problem -- rather than a fundamental flaw in the direct-
# surface-learning idea itself.
#
# Known limitations of this implementation, for whoever extends it:
# - MAX_PER_GROUP=80 subsampling (cell-4) discards most of each group's raw
#   order statistics before flattening -- chosen to keep the flattened
#   dataset a manageable size for training, but means the model never sees
#   the very densest/most granular parts of any single group's empirical
#   CDF; if fit quality (cell-8) is noticeably worse than the parametric-
#   family models, retrying with a higher MAX_PER_GROUP (or no subsampling
#   at all, at the cost of a much larger flattened dataset and slower
#   training) is the first thing to try.
# - The physics penalty (cell-7) is evaluated only at actual TRAINING rows'
#   own (theta,p,logt) locations, not at additional randomly-sampled
#   "collocation points" spanning the full input domain (the more complete
#   version of this technique, as used in physics-informed neural networks)
#   -- meaning regions of (theta,p,logt) space far from any real training
#   point have no enforcement at all, which is very likely why cell-9's
#   grid-based violation check didn't clearly improve. Adding a separate
#   collocation-point sampling step (drawing random theta/p/logt
#   combinations, independent of whether real data exists there, and
#   applying only the penalty terms -- no data_loss term -- at those
#   points) would be a natural next extension, and is standard practice in
#   the physics-informed NN literature for exactly this reason.
# - LAM_T=LAM_P=1.0 (cell-7) were not tuned -- picked as a reasonable
#   starting point, matching the untuned lam=1.0 used in the skew-logistic
#   notebook's monotonicity experiment, not the result of a sweep.
# - Single random train/val split and single network initialization per
#   model (same caveat as every other model-comparison notebook in this
#   project) -- the param-set-88 failure in particular should be checked
#   against at least one more random split before concluding it's a
#   structural weakness of the approach rather than one unlucky run.